In [29]:
from dlshogi.network.policy_value_network import policy_value_network
from dlshogi import serializers
from pydlshogi2.dataloader import HcpeDataLoader
import torch
import numpy as np
import sqlite3
import pickle
from typing import Sequence, Dict, Tuple, Union

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batchsize = 512

train_data = "/workspace/test2.hcpe"
arg_network = "kifcaption"
arg_model = "/workspace/model/model_resnet10_swish-072_for_caption"
word_to_id_file = "/workspace/kif_caption/word_to_id.pkl"

model = policy_value_network(arg_network)
model.to(device)
serializers.load_npz(arg_model, model)



In [31]:
from utils.token_move_tool import words_with_shogi_move
'''
トークナイザ - 文章(caption)を単語IDのリスト(tokens_id)に変換
caption   : 画像キャプション
word_to_id: 単語->単語ID辞書
'''
def tokenize_caption(caption: str, word_to_id: Dict[str, int]):
    tokens = words_with_shogi_move(caption)
    
    tokens_temp = []    
    # 単語についたピリオド、カンマを削除
    for token in tokens:
        if token in {'。', '、', '.', ',', '！', '？', '!', '?'}: 
            continue
        
        tokens_temp.append(token)
    
    tokens = tokens_temp        
        
    # 文章(caption)を単語IDのリスト(tokens_id)に変換
    tokens_ext = ['<start>'] + tokens + ['<end>']
    tokens_id = []
    for k in tokens_ext:
        if k in word_to_id:
            tokens_id.append(word_to_id[k])
        else:
            tokens_id.append(word_to_id['<unk>'])
    
    return torch.Tensor(tokens_id)

In [32]:
'''
batch     : features1, features2, move_label, result, コメントインデックスをまとめたもの
word_to_id: 単語->単語ID辞書
'''
def collate_func(x1, x2, move_label, result, index, word_to_id: Dict[str, int]):

    # SQLiteデータベースに接続
    conn = sqlite3.connect("/workspace/kif_caption/comments.db")
    cursor = conn.cursor()

    # index のすべての要素に対して一度に SQL クエリを実行
    index_values = [idx.item() for idx in index]
    placeholders = ', '.join(['?'] * len(index_values))  # SQLクエリ用のプレースホルダ
    cursor.execute(f'SELECT * FROM comments WHERE comment_index IN ({placeholders})', tuple(index_values))
    rows = cursor.fetchall()

    # 取得したデータを使ってトークナイズ
    captions = []
    for row in rows:
        caption = row[1]  # キャプションが row[1] にあると仮定
        captions.append(tokenize_caption(caption, word_to_id))

    # 接続を閉じる
    conn.close()
    
    # キャプションの長さが降順になるように並び替え
    batch = zip(x1, x2, move_label, result, captions)
    batch = sorted(batch, key=lambda x: len(x[4]), reverse=True)
    x1, x2, move_label, result, captions = zip(*batch)
    x1 = torch.stack(x1)
    x2 = torch.stack(x2)
    move_label = torch.stack(move_label)
    result = torch.stack(result)

    lengths = [cap.shape[0] for cap in captions]
    targets = torch.full((len(captions), max(lengths)), 
                         word_to_id['<null>'], dtype=torch.int64)
    for i, cap in enumerate(captions):
        end = lengths[i]
        targets[i, :end] = cap[:end]
    
    return x1, x2, move_label, result, targets, lengths

In [33]:
train_dataloader = HcpeDataLoader(train_data, batchsize, device, shuffle=True)

In [ ]:
# 学習のデバッグ用

# 辞書（単語→単語ID）の読み込み
with open(word_to_id_file, 'rb') as f:
    word_to_id = pickle.load(f)

for x1, x2, move_label, result, index in train_dataloader:
    x1, x2, move_label, result, targets, lengths = collate_func(x1, x2, move_label, result, index, word_to_id)

    y1, y2, policy_hidden, value_hidden, resnet_hidden = model(x1, x2)
    policy_features = torch.flatten(policy_hidden, 1)
    value_features = torch.flatten(value_hidden, 1)
    features = torch.cat([policy_features, value_features], dim=-1)
    print(x1.shape)
    print(x2.shape)
    print(move_label.shape)
    print(result.shape)
    print(targets[0])
    print(type(lengths))

    break

torch.Size([512, 62, 9, 9])
torch.Size([512, 57, 9, 9])
torch.Size([512])
torch.Size([512, 1])
tensor([20272, 13723,  6217, 14027,  6107,  9506,  5623, 16770,  4968, 10603,
         6925,  5894,  9163, 14671, 19970,  5907, 14027,  6916,  5622,  5936,
         6107,  2377,  6175,  6875,  6107, 20107,  6175, 10114,  4968, 11481,
        11357,  5907, 10684, 16003,  5907,  6727,  4530, 14027,  6175, 17098,
        13332,  6217,  4229,  4968, 13412,  5907,  1248,   855,  9838, 10865,
         6091,  4114,  4968, 15970,  1248,   855,  9838, 10885,  6091, 15885,
         6175,  7421,  8177,  4968, 19469,  5894, 11838,  4968, 12677, 10372,
         5510,    31, 14256, 16219,  6217,  9711,  6175, 14027, 12834,  6175,
        10681,  5922,  4558, 14851,  5724,  5622,   256, 14256, 16219,  6217,
        15602, 14047,  6107,  6097,  5159,  4968,  9140,  5223,  6927, 19898,
         6107, 13332,  6963,  6832,    31, 14256, 16219,  4937, 19623, 12185,
         4968, 17100,  5894,  4652,  5697, 1608

In [6]:
import sqlite3

# SQLiteデータベースに接続
conn = sqlite3.connect("/workspace/kif_caption/comments.db")
cursor = conn.cursor()

# コメントテーブルの件数を取得
cursor.execute('SELECT COUNT(*) FROM comments')
count = cursor.fetchone()[0]  # 結果から件数を取得

print(f"データベースに登録されているコメントの件数: {count}")

# コメントの取得
cursor.execute('SELECT * FROM comments WHERE comment_index = ?', (179,))
row = cursor.fetchone()

if row:
    print(f"インデックス {row[0]} のコメント: {row[1]}")
else:
    print("コメントが見つかりません")

# 接続を閉じる
conn.close()

データベースに登録されているコメントの件数: 141379
インデックス 179 のコメント: 互いに駒に当てる手が続いた。△7六飛には▲6五角がある。△1六飛と交換を迫ってどうか。飛車が手に入れば、後手は△4七角が楽しみだ。検討の結果、△1六歩には▲1八飛△3四飛▲1六飛の順で先手が指せるようだ。以下、(1)△4五桂は▲4六銀△3八角▲4五銀△1六角成▲同香△3五飛に、▲5六角か▲5六銀。後手は持ち歩が少ないため攻めを続けづらい。後手「歩が足りていないですね。じゃあ、(68手目△4六歩と)取り込んでからダメなんですね」


In [5]:
import numpy as np
dtypeHcp = np.dtype((np.uint8, 32))
dtypeEval = np.dtype(np.int16)
dtypeMove16 = np.dtype(np.int16)
dtypeGameResult = np.dtype(np.int8)

HuffmanCodedPosAndEvalComment = np.dtype(
    [('hcp', dtypeHcp),
     ('eval', dtypeEval),
     ('bestMove16', dtypeMove16),
     ('gameResult', dtypeGameResult),
     ('dummy', np.uint8),
     ('comment_index', np.int32),
	])

def load_hcpe_file(filepath):
    # ファイルを HuffmanCodedPosAndEval の形式で読み取る
    data = np.fromfile(filepath, dtype=HuffmanCodedPosAndEvalComment)
    return data

In [17]:
# 使用例
hcpe_data = load_hcpe_file("/workspace/train1.hcpe")

# データの内容を確認
for i, record in enumerate(hcpe_data[:10]):  # 最初の10レコードを表示
    print(record["comment_index"])

0
1
2
2
3
4
5
6
7
8
